# Notebook 1 — Préparation des données IMDb (raw → processed)

## Objectif
Ce notebook prépare une version **propre, exploitable et reproductible** du dataset IMDb pour le projet de **prédiction du genre de films avec Naive Bayes**.

## Logique retenue
Le sujet impose une prédiction du genre à partir de variables numériques. Or le dataset IMDb Top 1000 contient un champ `Genre` souvent **multi-étiquettes** (`"Action, Crime, Drama"`), ce qui n'est pas directement compatible avec une classification simple dans sa forme la plus élémentaire.

Le pipeline de préparation retenu est donc le suivant :

1. chargement du dataset brut ;
2. contrôle de structure et de qualité ;
3. sélection des variables utiles au sujet ;
4. traitement des valeurs manquantes ;
5. réduction du champ `Genre` à son **genre principal** ;
6. limitation aux genres les plus représentés pour éviter des classes extrêmement rares ;
7. encodage de la variable cible ;
8. export d'un dataset `processed` prêt pour l'entraînement.

## Fichiers attendus dans le projet
Ce notebook suppose l'organisation suivante :

```text
projet-naive-bayes-imdb/
├── data/
│   ├── raw/
│   │   └── imdb_top_1000.csv
│   └── processed/
├── notebooks/
└── src/
```

Le notebook est donc conçu pour être placé dans `notebooks/`.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## 1. Définition des chemins

On utilise des chemins relatifs pour garder le notebook portable au sein du projet.


In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "imdb_top_1000.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH, PROCESSED_DIR

(WindowsPath('C:/Users/KenziLali/Desktop/projet-naive-bayes-imdb/data/raw/imdb_top_1000.csv'),
 WindowsPath('C:/Users/KenziLali/Desktop/projet-naive-bayes-imdb/data/processed'))

## 2. Chargement du dataset brut

In [3]:
df_raw = pd.read_csv(RAW_PATH)
df_raw.head()

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,Drama,9.300,Two imprisoned men bond over a number of years...,80.000,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,"28,341,469"
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"Crime, Drama",9.200,An organized crime dynasty's aging patriarch t...,100.000,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.000,When the menace known as the Joker wreaks havo...,84.000,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202 min,"Crime, Drama",9.000,The early life and career of Vito Corleone in ...,90.000,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,"57,300,000"
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96 min,"Crime, Drama",9.000,A jury holdout attempts to prevent a miscarria...,96.000,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,"4,360,000"


In [4]:
print(f"Dimensions : {df_raw.shape[0]} lignes × {df_raw.shape[1]} colonnes")
df_raw.info()

Dimensions : 1000 lignes × 16 colonnes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Poster_Link    1000 non-null   object 
 1   Series_Title   1000 non-null   object 
 2   Released_Year  1000 non-null   object 
 3   Certificate    899 non-null    object 
 4   Runtime        1000 non-null   object 
 5   Genre          1000 non-null   object 
 6   IMDB_Rating    1000 non-null   float64
 7   Overview       1000 non-null   object 
 8   Meta_score     843 non-null    float64
 9   Director       1000 non-null   object 
 10  Star1          1000 non-null   object 
 11  Star2          1000 non-null   object 
 12  Star3          1000 non-null   object 
 13  Star4          1000 non-null   object 
 14  No_of_Votes    1000 non-null   int64  
 15  Gross          831 non-null    object 
dtypes: float64(2), int64(1), object(13)
memory usage: 125.1+ K

## 3. Contrôle initial de qualité

On vérifie :
- les noms de colonnes ;
- les types ;
- les valeurs manquantes ;
- la cohérence avec l'énoncé du projet.

### Colonnes du dataset utiles pour ce projet
L'énoncé mentionne explicitement :
- le nombre de votes ;
- une mesure de popularité ;
- la note moyenne ;
- le genre.

Dans ce dataset, l'équivalent le plus pertinent disponible est :
- `No_of_Votes` pour le nombre de votes ;
- `IMDB_Rating` pour la note IMDb ;
- `Meta_score` comme score externe complémentaire ;
- `Genre` comme variable cible.

Le dataset ne contient pas de colonne `Popularity` au sens strict. Le `Meta_score` jouera donc le rôle de troisième variable numérique explicative.


In [5]:
df_raw.columns.tolist()

['Poster_Link',
 'Series_Title',
 'Released_Year',
 'Certificate',
 'Runtime',
 'Genre',
 'IMDB_Rating',
 'Overview',
 'Meta_score',
 'Director',
 'Star1',
 'Star2',
 'Star3',
 'Star4',
 'No_of_Votes',
 'Gross']

In [6]:
missing = df_raw.isna().sum().sort_values(ascending=False)
missing.to_frame("nb_valeurs_manquantes")

,nb_valeurs_manquantes
Gross,169
Meta_score,157
Certificate,101
Poster_Link,0
Series_Title,0
Released_Year,0
Runtime,0
Genre,0
IMDB_Rating,0
Overview,0


## 4. Sélection des colonnes utiles

On restreint volontairement le dataset aux variables mobilisées dans le sujet afin de garder un pipeline cohérent avec l'énoncé.


In [7]:
selected_columns = ["No_of_Votes", "IMDB_Rating", "Meta_score", "Genre"]
df = df_raw[selected_columns].copy()
df.head()

,No_of_Votes,IMDB_Rating,Meta_score,Genre
0,2343110,9.300,80.000,Drama
1,1620367,9.200,100.000,"Crime, Drama"
2,2303232,9.000,84.000,"Action, Crime, Drama"
3,1129952,9.000,90.000,"Crime, Drama"
4,689845,9.000,96.000,"Crime, Drama"


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   No_of_Votes  1000 non-null   int64  
 1   IMDB_Rating  1000 non-null   float64
 2   Meta_score   843 non-null    float64
 3   Genre        1000 non-null   object 
dtypes: float64(2), int64(1), object(1)
memory usage: 31.4+ KB


## 5. Traitement des valeurs manquantes

`Meta_score` contient des valeurs manquantes.  
Dans ce notebook, on applique une stratégie simple et robuste pour un projet académique : **suppression des lignes incomplètes** sur les colonnes retenues.

Cette décision est défendable ici car :
- on travaille sur un dataset de taille raisonnable ;
- on souhaite conserver une préparation simple et traçable ;
- l'objectif principal est de préparer un dataset propre pour une première modélisation.


In [9]:
print("Valeurs manquantes avant nettoyage :")
display(df.isna().sum().to_frame("nb_na"))

df = df.dropna().reset_index(drop=True)

print("Valeurs manquantes après nettoyage :")
display(df.isna().sum().to_frame("nb_na"))
print(f"Dimensions après suppression des NaN : {df.shape}")

Valeurs manquantes avant nettoyage :


,nb_na
No_of_Votes,0
IMDB_Rating,0
Meta_score,157
Genre,0


Valeurs manquantes après nettoyage :


,nb_na
No_of_Votes,0
IMDB_Rating,0
Meta_score,0
Genre,0


Dimensions après suppression des NaN : (843, 4)


## 6. Analyse et simplification de la variable `Genre`

Le champ `Genre` est souvent multi-valué, par exemple :
- `Crime, Drama`
- `Action, Crime, Drama`

Or la version la plus simple du problème de classification supervisée attendue ici consiste à prédire **une seule classe** par observation.

### Choix méthodologique
On retient donc le **genre principal**, défini comme le premier genre de la chaîne.  
Exemple :
- `Action, Crime, Drama` → `Action`
- `Crime, Drama` → `Crime`

Ce choix simplifie la cible, tout en restant explicable dans le rapport.


In [10]:
df["Genre"].head(10)

0                        Drama
1                 Crime, Drama
2         Action, Crime, Drama
3                 Crime, Drama
4                 Crime, Drama
5     Action, Adventure, Drama
6                 Crime, Drama
7    Biography, Drama, History
8    Action, Adventure, Sci-Fi
9                        Drama
Name: Genre, dtype: object

In [11]:
df["Genre"] = df["Genre"].str.split(",").str[0].str.strip()
df["Genre"].head(10)

0        Drama
1        Crime
2       Action
3        Crime
4        Crime
5       Action
6        Crime
7    Biography
8       Action
9        Drama
Name: Genre, dtype: object

## 7. Étude de la distribution des genres

Avant entraînement, il faut vérifier si certaines classes sont trop rares.
Des classes très peu représentées dégradent fortement l'apprentissage et rendent les métriques peu fiables.


In [12]:
genre_counts = df["Genre"].value_counts()
genre_counts.to_frame("effectif")

,effectif
Genre,
Drama,241
Action,143
Comedy,125
Crime,87
Biography,79
Animation,75
Adventure,64
Horror,11
Mystery,8


## 8. Filtrage des genres les plus fréquents

On conserve ici les **5 genres principaux** :
- pour réduire le déséquilibre de classes ;
- pour rendre le problème plus stable ;
- pour rester sur une base suffisamment représentée.

Ce choix devra être explicitement justifié dans le rapport comme une décision de préparation destinée à améliorer la robustesse du modèle.


In [13]:
TOP_N_GENRES = 5
top_genres = df["Genre"].value_counts().nlargest(TOP_N_GENRES).index.tolist()
top_genres

['Drama', 'Action', 'Comedy', 'Crime', 'Biography']

In [14]:
df = df[df["Genre"].isin(top_genres)].reset_index(drop=True)
df["Genre"].value_counts().to_frame("effectif")

,effectif
Genre,
Drama,241
Action,143
Comedy,125
Crime,87
Biography,79


## 9. Encodage de la variable cible

Le modèle attend une cible numérique.  
On encode donc le genre avec `LabelEncoder`.


In [15]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df["Genre_encoded"] = label_encoder.fit_transform(df["Genre"])

label_mapping = pd.DataFrame({
    "Genre": label_encoder.classes_,
    "Genre_encoded": range(len(label_encoder.classes_))
}).sort_values("Genre_encoded").reset_index(drop=True)

label_mapping

,Genre,Genre_encoded
0,Action,0
1,Biography,1
2,Comedy,2
3,Crime,3
4,Drama,4


## 10. Contrôles finaux sur le dataset préparé

In [16]:
print(f"Dimensions finales : {df.shape[0]} lignes × {df.shape[1]} colonnes")
df.head()

Dimensions finales : 675 lignes × 5 colonnes


,No_of_Votes,IMDB_Rating,Meta_score,Genre,Genre_encoded
0,2343110,9.300,80.000,Drama,4
1,1620367,9.200,100.000,Crime,3
2,2303232,9.000,84.000,Action,0
3,1129952,9.000,90.000,Crime,3
4,689845,9.000,96.000,Crime,3


In [17]:
df.describe(include="all")

,No_of_Votes,IMDB_Rating,Meta_score,Genre,Genre_encoded
count,675.000,675.000,675.000,675,675.000
unique,NaN,NaN,NaN,5,NaN
top,NaN,NaN,NaN,Drama,NaN
freq,NaN,NaN,NaN,241,NaN
mean,"311,776.858",7.929,77.446,NaN,2.302
std,"354,327.945",0.288,12.562,NaN,1.560
min,"25,198.000",7.600,28.000,NaN,0.000
25%,"68,382.500",7.700,70.000,NaN,1.000
50%,"171,739.000",7.900,78.000,NaN,2.000
75%,"435,384.500",8.100,86.000,NaN,4.000


## 11. Export des données `processed`

Le notebook d'entraînement utilisera exclusivement ces fichiers :
- `imdb_genre_nb_ready.csv` : dataset final prêt à l'emploi ;
- `genre_label_mapping.csv` : table de correspondance entre labels texte et labels numériques.


In [18]:
processed_path = PROCESSED_DIR / "imdb_genre_nb_ready.csv"
mapping_path = PROCESSED_DIR / "genre_label_mapping.csv"

df.to_csv(processed_path, index=False)
label_mapping.to_csv(mapping_path, index=False)

print(f"Dataset processed enregistré : {processed_path}")
print(f"Mapping des labels enregistré : {mapping_path}")

Dataset processed enregistré : C:\Users\KenziLali\Desktop\projet-naive-bayes-imdb\data\processed\imdb_genre_nb_ready.csv
Mapping des labels enregistré : C:\Users\KenziLali\Desktop\projet-naive-bayes-imdb\data\processed\genre_label_mapping.csv


## 12. Conclusion de préparation

À l'issue de ce notebook :
- les colonnes utiles au sujet ont été isolées ;
- les valeurs manquantes ont été traitées ;
- le champ `Genre` a été simplifié en cible mono-classe ;
- les classes très rares ont été écartées ;
- la variable cible a été encodée ;
- le dataset final a été exporté vers `data/processed/`.

Le notebook suivant pourra maintenant se concentrer sur :
- le chargement de la donnée `processed` ;
- la séparation entraînement / test ;
- l'entraînement du modèle Naive Bayes ;
- l'évaluation des performances.
